In [1]:
import os
import pandas as pd
import numpy as np
from collections import Counter
from PIL import Image  # for extracting image dimensions and brightness

train_csv_path = "/kaggle/input/cassava-leaf-disease-classification/train.csv"
test_image_dir = "/kaggle/input/cassava-leaf-disease-classification/test_images"
train_image_dir = "/kaggle/input/cassava-leaf-disease-classification/train_images"

train_df = pd.read_csv(train_csv_path)

label_sizes = {}
label_widths = {}
label_heights = {}
label_dims = {}
label_brightness = {}
dim_sample_limit = 5000
dim_counts = Counter()
brightness_sample_limit = 2000
brightness_counts = Counter()

for _, row in train_df.iterrows():
    img_path = os.path.join(train_image_dir, row["image_id"])
    label = int(row["label"])
    try:
        size = os.path.getsize(img_path)
        label_sizes.setdefault(label, []).append(size)
    except FileNotFoundError:
        continue

    if dim_counts[label] < dim_sample_limit:
        try:
            with Image.open(img_path) as im:
                width, height = im.size
                label_widths.setdefault(label, []).append(width)
                label_heights.setdefault(label, []).append(height)
                label_dims.setdefault(label, []).append(width * height)
                dim_counts[label] += 1
        except Exception:
            pass  # ignore unreadable images

    if brightness_counts[label] < brightness_sample_limit:
        try:
            with Image.open(img_path) as im:
                gray = im.convert("L")
                arr = np.array(gray)
                brightness = arr.mean()
                label_brightness.setdefault(label, []).append(brightness)
                brightness_counts[label] += 1
        except Exception:
            pass

all_sizes = [s for sizes in label_sizes.values() for s in sizes]
overall_mean = np.mean(all_sizes) if all_sizes else 0.0
overall_std = np.std(all_sizes) if all_sizes else 1.0

all_dims = [d for dims in label_dims.values() for d in dims]
overall_mean_dim = np.mean(all_dims) if all_dims else 0.0
overall_std_dim = np.std(all_dims) if all_dims else 1.0

all_widths = [w for ws in label_widths.values() for w in ws]
overall_mean_w = np.mean(all_widths) if all_widths else 0.0
overall_std_w = np.std(all_widths) if all_widths else 1.0

all_heights = [h for hs in label_heights.values() for h in hs]
overall_mean_h = np.mean(all_heights) if all_heights else 0.0
overall_std_h = np.std(all_heights) if all_heights else 1.0

all_brightness = [b for bs in label_brightness.values() for b in bs]
overall_mean_b = np.mean(all_brightness) if all_brightness else 0.0
overall_std_b = np.std(all_brightness) if all_brightness else 1.0

label_mean_size = {
    lbl: (np.mean(sizes) if sizes else overall_mean)
    for lbl, sizes in label_sizes.items()
}
label_std_size = {
    lbl: (np.std(sizes) if sizes else overall_std) for lbl, sizes in label_sizes.items()
}

label_mean_dim = {
    lbl: (np.mean(dims) if dims else overall_mean_dim)
    for lbl, dims in label_dims.items()
}
label_std_dim = {
    lbl: (np.std(dims) if dims else overall_std_dim) for lbl, dims in label_dims.items()
}

label_mean_w = {
    lbl: (np.mean(ws) if ws else overall_mean_w) for lbl, ws in label_widths.items()
}
label_std_w = {
    lbl: (np.std(ws) if ws else overall_std_w) for lbl, ws in label_widths.items()
}

label_mean_h = {
    lbl: (np.mean(hs) if hs else overall_mean_h) for lbl, hs in label_heights.items()
}
label_std_h = {
    lbl: (np.std(hs) if hs else overall_std_h) for lbl, hs in label_heights.items()
}

label_mean_b = {
    lbl: (np.mean(bs) if bs else overall_mean_b) for lbl, bs in label_brightness.items()
}
label_std_b = {
    lbl: (np.std(bs) if bs else overall_std_b) for lbl, bs in label_brightness.items()
}

label_counts = Counter(train_df["label"])
total_labels = sum(label_counts.values())
label_prior = {lbl: cnt / total_labels for lbl, cnt in label_counts.items()}

most_common_label = int(label_counts.most_common(1)[0][0])

print("Per‑label stats computed.")




Per‑label stats computed.


In [2]:
image_predictions = []
# Strengthen the prior bias (more negative) to favour frequent classes
alpha = -1.0  # increased magnitude compared to the original -0.3

for image_id in os.listdir(test_image_dir):
    if not image_id.lower().endswith((".jpg", ".jpeg", ".png")):
        continue
    test_path = os.path.join(test_image_dir, image_id)

    try:
        test_size = os.path.getsize(test_path)
    except FileNotFoundError:
        test_size = overall_mean

    try:
        with Image.open(test_path) as im:
            test_w, test_h = im.size
            test_dim = test_w * test_h
            gray = im.convert("L")
            arr = np.array(gray)
            test_b = arr.mean()
    except Exception:
        test_w, test_h = overall_mean_w, overall_mean_h
        test_dim = overall_mean_dim
        test_b = overall_mean_b

    best_label = min(
        label_mean_size.keys(),
        key=lambda lbl: (
            ((test_size - label_mean_size[lbl]) / (label_std_size[lbl] + 1e-6)) ** 2
            + ((test_dim - label_mean_dim[lbl]) / (label_std_dim[lbl] + 1e-6)) ** 2
            + ((test_w - label_mean_w[lbl]) / (label_std_w[lbl] + 1e-6)) ** 2
            + ((test_h - label_mean_h[lbl]) / (label_std_h[lbl] + 1e-6)) ** 2
            # Brightness term removed as it adds noise
            + alpha * np.log(label_prior.get(lbl, 1e-6))
        ),
    )
    image_predictions.append({"image_id": image_id, "label": int(best_label)})

submission_df = pd.DataFrame(image_predictions)




In [3]:
submission_path = "/kaggle/working/submission.csv"
submission_df.to_csv(submission_path, index=False)
print(f"Submission file saved at: {submission_path}")




Submission file saved at: /kaggle/working/submission.csv


In [4]:
print(submission_df.head())

         image_id  label
0  2574872277.jpg      3
1  1449210447.jpg      3
2  1269550303.jpg      3
3  1411799307.jpg      3
4  2474336811.jpg      1
